In [1]:
import os
import time

import pycuda.autoinit
from utils.yolo_classes import get_cls_dict
from utils.display import open_window, set_display, show_fps
from utils.visualization import BBoxVisualization
from utils.yolo import TRT_YOLO

# 🚨 重要提醒：
# 這裡的 "yolov4-tiny-416" 代表系統會去尋找名為 yolov4-tiny-416.trt 的引擎檔。
# 請務必先使用轉換腳本，將您剛剛訓練好的 _best.weights 轉成 .onnx，再轉成 .trt 檔！
trt_yolo = TRT_YOLO("yolov4-tiny-416", (416, 416), 4)

In [2]:
import cv2
import ipywidgets.widgets as widgets
from IPython.display import display

# 1. 讀取影像
img = cv2.imread('All1.jpg')

# 2. 進行模型辨識
boxes, confs, clss = trt_yolo.detect(img, 0.3)

# 3. 跑迴圈將辨識結果畫在影像上
for i in range(len(clss)):
    cv2.rectangle(img, (boxes[i][0], boxes[i][1]), (boxes[i][2], boxes[i][3]), (255, 0, 0), 2)

# 將 OpenCV 的影像編碼成 JPEG 格式
_, encoded_image = cv2.imencode('.jpg', img)

# 建立一個 Image 元件並顯示
image_widget = widgets.Image(value=encoded_image.tobytes(), format='jpg', width=400)
display(image_widget)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [3]:
import traitlets
from IPython.display import display
import ipywidgets.widgets as widgets
from jetbot import Camera, bgr8_to_jpeg
from jetbot import Robot

# 只有使用 USBCamera 的需要這幾行
# from jetcam.usb_camera import USBCamera
# camera = USBCamera(capture_device=0)
# camera.running = True

# 一般 camera (CSI 介面) 的用這一行
camera = Camera.instance(width=416, height=416)

image = widgets.Image(format='jpeg', width=224, height=224)
camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

robot = Robot()

In [4]:
import numpy as np

def getROI(img, vertices):
    mask = np.zeros_like(img) # 建立遮罩
    cv2.fillPoly(mask, vertices, (255,)) # 把多邊形填滿白色
    masked_image = cv2.bitwise_and(img, mask)
    return masked_image # 找要跟蹤的點

def make_points(image, average):
    slope, y_int = average
    y1 = image.shape[0]
    y2 = int(y1 * (1/6))
    x1 = int((y1 - y_int) // slope)
    x2 = int((y2 - y_int) // slope)
    return np.array([x1, y1, x2, y2])

In [5]:
def average(image, lines):
    if lines is None: # 將多條線中畫出左右各一條線
        return
    left = []
    right = []
    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)
        parameters = np.polyfit((x1, x2), (y1, y2), 1)
        slope = parameters[0]
        y_int = parameters[1]
        if slope < 0:
            left.append((slope, y_int))
        else:
            right.append((slope, y_int))

    right_avg = np.average(right, axis=0)
    left_avg = np.average(left, axis=0)

    if type(left_avg) is np.float64:
        right_line = make_points(image, right_avg)
        return np.array([right_line,])
    if type(right_avg) is np.float64:
        left_line = make_points(image, left_avg)
        return np.array([left_line,])

    right_line = make_points(image, right_avg)
    left_line = make_points(image, left_avg)
    return np.array([left_line, right_line])

In [6]:
def getAverageLines(origin_road):
    # 將影像縮放為 640x480 (配合原有的車道座標邏輯)
    road_resized = cv2.resize(origin_road, (640, 480))

    # 讀入 RGB 影像轉換為灰階景
    road_gray = cv2.cvtColor(road_resized, cv2.COLOR_BGR2GRAY)

    # 濾波：使用高斯模糊去除雜訊 (加大模糊程度，把地板紋路糊掉，保留白線)
    road_gray = cv2.GaussianBlur(road_gray, (9, 9), 0)

    # 🚨 修正 1：將 Canny 門檻調回正常人類視覺範圍
    edges = cv2.Canny(road_gray, 50, 150)

    # 🚨 修正 2：修改 ROI 頂點，切掉畫面上半部的「教室背景」，只保留地板
    ROI_vertices = [(0, 480), (0, 240), (200, 150), (440, 150), (640, 240), (640, 480)]

    # 僅保留 ROI 區域內的邊緣
    edges = getROI(edges, np.array([ROI_vertices], np.int32))

    # 直線偵測：稍微放寬偵測條件，讓虛線也能被連起來
    linesp = cv2.HoughLinesP(edges, rho=2, theta=np.pi / 180,
                             threshold=50, minLineLength=30, maxLineGap=15)

    # 回傳平均化後的左右車道線
    return average(road_resized, linesp)

In [13]:
def modifySpeed(origin_road):
    global speed_left, speed_right, speed

    # 取得處理後的左右車道線平均座標
    averaged_lines = getAverageLines(origin_road)

    if averaged_lines is not None:
        # 計算偵測到的車道線中點的 X 座標
        mid_x = int(np.average(averaged_lines, axis=0)[2])
        
        target_center = 300

        # 根據中點位置判斷偏向並調整速度
        if mid_x < (target_center - 15):
            # 中點偏左，增加右輪速度以向左修正
            speed_left = speed
            speed_right = speed + 0.03
        elif mid_x > (target_center + 15):
            # 中點偏右，增加左輪速度以向右修正
            speed_left = speed + 0.03
            speed_right = speed
        else:
            # 接近正中央，微調左輪維持穩定
            speed_left = speed 
            speed_right = speed

In [8]:
import time

# 全域變數
stop_ignore = 0
railway_ignore = 0
count = 0
stop_until_time = 0

# 🚨 新增：防止塞車的「處理中」鎖定變數
is_processing = False 

def getNearest(signs):
    # 如果沒有偵測到任何標誌，回傳一個無害的預設值 [寬度0, 假ID -1]
    # (不要回傳 3，因為 3 現在是 Stop，會導致誤判！)
    if not len(signs):
        return [0, -1]

    t = time.time()
    cls_id = signs[0][1]

    # 🚨 根據您最新的 Class ID 重新設定冷卻時間！
    # Stop 現在是 3，Railway 現在是 2
    if (cls_id == 3 and (t > stop_ignore)) \
        or (cls_id == 2 and (t > railway_ignore)) \
        or (cls_id in [0, 1]):  # road close(0) 和 slow(1) 沒有冷卻時間限制
        return signs[0]

    # 如果正在冷卻中，就遞迴檢查畫面中的下一個標誌
    return getNearest(signs[1:]) if len(signs) >= 2 else [0, -1]

In [9]:
def update(change):
    global robot, count, speed_left, speed_right, stop_ignore, railway_ignore, stop_until_time, is_processing
    
    # 🚨 核心機制：如果上一張照片還沒處理完，直接把新照片丟進垃圾桶！
    if is_processing:
        return
    
    # 把鎖關上，表示「我現在很忙」
    is_processing = True 

    try:
        t = time.time()

        # 防撞牆機制：檢查是否還在「必須停止」的時間內
        if t < stop_until_time:
            robot.stop()
            return # 注意：這裡 return 會自動跳到最下面的 finally 把鎖解開

        img = change['new']
        
        # 執行車道追蹤與速度修正
        modifySpeed(img) 

        # 降低 YOLO 偵測頻率
        if count < 2:
            count += 1
            return
        else:
            count = 0

        # 偵測路牌
        boxes, confs, clss = trt_yolo.detect(img)

        signs = []
        for box, cls in zip(boxes, clss):
            width = box[2] - box[0]
            signs.append([width, cls])

        # 找出最靠近車子的路牌
        signs.sort(reverse=True, key=lambda x : x[0])
        sign = getNearest(signs)
        ALERT_WIDTH = 30

        # 根據標誌類型執行動作
        if sign[1] == 1 and sign[0] > ALERT_WIDTH:
            print('slow')
            robot.set_motors(speed_left * 0.7, speed_right * 0.7)
        elif (sign[1] == 0 and sign[0] > ALERT_WIDTH):
            print("road close")
            robot.stop()
        elif sign[1] == 3 and sign[0] > ALERT_WIDTH:
            print("stop")
            robot.stop()
            stop_ignore = t + 10
            stop_until_time = t + 3
        elif sign[1] == 2 and sign[0] > (ALERT_WIDTH - 20):
            print("railway")
            robot.stop()
            railway_ignore = t + 10
            stop_until_time = t + 5
        else:
            robot.set_motors(speed_left, speed_right)

    finally:
        # 🚨 關鍵：不管上面發生什麼事，處理完了就把鎖解開，允許接收下一張最新照片
        is_processing = False

In [10]:
import cv2
import ipywidgets.widgets as widgets
from IPython.display import display
import numpy as np

# 1. 驗證鏡頭畫面 (這裡是 416x416)
img = camera.value

# 2. 偵測路牌標誌
boxes, confs, clss = trt_yolo.detect(img, 0.3)

# 3. 取得車道平均線
averaged_lines = getAverageLines(img)

if averaged_lines is not None:
    for x1, y1, x2, y2 in averaged_lines:
        # 🚨 修正 3：座標正確縮放！
        # 把 640x480 的數學座標，正確投影到 416x416 的實際畫布上
        cv2.line(img, (int(x1 / 640 * 416), int(y1 / 480 * 416)),
                 (int(x2 / 640 * 416), int(y2 / 480 * 416)), (0, 0, 255), 5)

    # 計算並畫出路徑中點的綠色標記
    mid_x = int(np.average(averaged_lines, axis=0)[2])
    # 中點標記也要改成 / 640 * 416
    cv2.line(img, (int(mid_x / 640 * 416), 180), (int(mid_x / 640 * 416), 220), (0, 255, 0), 4)
    print('mid: ', mid_x)
else:
    print("❌ 找不到車道線！(可能偏離軌道或光線太暗)")

# 4. 畫出路牌的辨識框
for i in range(len(clss)):
    cv2.rectangle(img, (boxes[i][0], boxes[i][1]), (boxes[i][2], boxes[i][3]), (0, 0, 255), 2)

# 5. 使用 ipywidgets 顯示
_, encoded_image = cv2.imencode('.jpg', img)
image_widget = widgets.Image(value=encoded_image.tobytes(), format='jpg', width=416, height=416)

display(image_widget)

mid:  310


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [14]:
import time

# 🚗 1. 設定合理的巡航與基準速度 (給予足夠扭力)
speed = 0.2        
speed_left = 0.20   
speed_right = 0.20 

# 🚗 2. 車子起步時的速度（給予瞬間大推力克服靜摩擦力）
robot.set_motors(0.30, 0.30) 
time.sleep(0.1)

# 🚗 3. 設定回我們微調過的巡航速度
robot.set_motors(speed_left, speed_right)

print("🚗 引擎啟動，進入自動駕駛模式！")

# 🚨 保險起見：先解除可能還在背景執行的 observe 監聽，避免畫面打架
try:
    camera.unobserve(update, names='value')
except:
    pass

# 🚗 4. 改回無窮迴圈的格式，方便您除錯與觀看辨識狀況
while(1):
    # 手動讀取當下最新的畫面送給 update 函數
    update({'new' : camera.value})
    
    # 🚨 關鍵保護機制：加入這行極短的延遲！
    # 這能讓 CPU 喘口氣，Jupyter 網頁才有「時間」把畫好框框的最新照片顯示在螢幕上。
    time.sleep(0.03)

🚗 引擎啟動，進入自動駕駛模式！
slow
slow
railway


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned


slow


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned


slow


...

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned


slow
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road close
road 

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned


railway


KeyboardInterrupt: 

In [16]:
# 將馬達動力歸零
speed = speed_left = speed_right = 0
robot.stop()
print("🛑 車輛已安全停止")

🛑 車輛已安全停止


In [ ]:
camera_link.unlink()

In [14]:
camera.stop()